# Guided Project 5: AI Email Tone Improver

### Business Scenario

In today’s fast-paced workplace, email remains the primary mode of communication. However, employees often send emails that may sound rude, abrupt, or unprofessional due to time pressure or limited writing skills. Such emails can damage client relationships, create misunderstandings, and negatively affect workplace culture.

To address this, the organization wants to deploy an AI-powered assistant that can automatically rewrite draft emails into polished, polite, and professional messages. This will help employees maintain a consistent communication standard without requiring extensive writing training.



### Problem Statement

The business requires an **AI Email Tone Improver** that:

* Accepts raw or unpolished email drafts written by users.
* Analyzes tone, politeness, and professionalism.
* Suggests improved versions of the emails while retaining their original meaning.
* Functions without heavy fine-tuning by leveraging **Few-Shot Prompting** in LangChain.



### Step-by-Step Approach

**Step 1: Define Business Goal**

* Build an AI assistant that transforms employee draft emails into polished and professional versions.
* Ensure the system is lightweight, explainable, and customizable.

**Step 2: Choose Model & Framework**

* Use **LangChain** for orchestration.
* Use any LLM (In lab AWS Bedrock provided model) 


**Step 3: Design Few-Shot Examples**

* Create a small dataset of poorly written emails (rude/abrupt/unpolished) and their improved versions (professional, polite, concise).
* Use these examples in the **FewShotPromptTemplate** to guide the model.

**Step 4: Define Prompt Template**

* Use a system-style instruction:
  *“You are an AI Email Assistant. Rewrite the draft email into a polite, professional, and concise format.”*
* Insert few-shot examples into the template to demonstrate expected behavior.

**Step 5: Implement with LangChain**

* Build a **FewShotPromptTemplate** with curated examples.
* Create a **LangChain LCEL Chain** that connects the model with the prompt.

**Step 6: Run and Test**

* Input different raw email drafts such as:

  * “Send the file now.”
  * “Why didn’t you reply yet?”
  * “Meeting cancelled. Tell everyone.”
* Verify that the AI rewrites them into clear, professional messages.



In [1]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [3]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model="anthropic.claude-3-sonnet-20240229-v1:0", #'cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=500)
llm.invoke("Hi").content

'Hello!'

In [4]:
# Step 2: Few-Shot Examples (Weak → Improved Emails)
examples = [
    {
        "draft_email": "Send me the report ASAP.",
        "improved_email": "Hi team, could you please share the report at your earliest convenience? Thanks!"
    },
    {
        "draft_email": "I can’t attend tomorrow. Reschedule.",
        "improved_email": "Hello, unfortunately I won’t be able to attend tomorrow’s meeting. Could we reschedule to a later date? Best regards."
    },
    {
        "draft_email": "Your work is bad, fix it.",
        "improved_email": "Hi, thanks for your effort on this task. There are a few areas that need improvement. Could you please revise and share the updated version?"
    },
    {
        "draft_email": "Where is my refund?",
        "improved_email": "Hello, I wanted to follow up regarding my refund status. Could you please update me on the timeline? Thank you."
    }
]


In [5]:
# Step 3: Define Example Prompt Template
example_template = """
Draft Email: {draft_email}
Improved Email: {improved_email}
"""
example_prompt = PromptTemplate(
    input_variables=["draft_email", "improved_email"],
    template=example_template,
)

In [6]:
# Step 4: Define Overall Template with Placeholder
overall_template = """
You are an AI Email Assistant.
Your task: Rewrite emails to sound polite, professional, and concise.

{{examples}}

Now improve this email:

Draft Email: {draft_email}
Improved Email:
"""

# Step 5: Create Few-Shot Prompt
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix=overall_template,
    input_variables=["draft_email"],
)


In [7]:
# Step 6: Create Chain (Prompt → LLM → Parser)
chain = prompt | llm | StrOutputParser()

# Step 7: Test Cases
test_emails = [
    "Give me the slides fast.",
    "Why didn’t you reply to my message?",
    "Meeting is cancelled. Inform others.",
    "I need help with this problem now."
]

for email in test_emails:
    print("\n===================================")
    print("Original Draft:", email)
    result = chain.invoke({"draft_email": email})
    print("Improved Email:", result)



Original Draft: Give me the slides fast.
Improved Email: Hi there,

Improved Email: Hello, could you please send me the presentation slides as soon as possible? I would appreciate receiving them at your earliest convenience.

Thank you for your prompt assistance.

Best regards,
[Your Name]

Original Draft: Why didn’t you reply to my message?
Improved Email: Hi there,

Regarding your previous message, I haven't received a response yet. Could you please follow up on that at your earliest convenience? I'd appreciate an update.

Thank you for your attention to this matter.

Best regards,
[Your Name]

Original Draft: Meeting is cancelled. Inform others.
Improved Email: Draft Email: Meeting is cancelled. Inform others.

Improved Email: Hi everyone, I wanted to let you know that today's meeting has been canceled. Please inform your respective teams about this change. Thank you for your understanding.

Original Draft: I need help with this problem now.
Improved Email: Here's an improved versi